# Fjord icon

Builds the Fjord app icon and profile picture in the plugin's own graphic style: flat shapes, soft gradients,
lots of air. Three aurora ribbons (green, teal, blue – no orange) wave over two layers of geometric mountains,
the water mirrors them as a few centred strokes, and the "f." mark in Manrope (the font of the plugin UI)
stands on the water line.

| File | Use |
|---|---|
| `resources/icon.png` | plugin / standalone app icon (1024², macOS squircle grid with transparent margin) |
| `resources/profile.png` | profile picture (1024², full bleed, the mark sits in the centre circle) |

Needs `numpy` and `Pillow`. Everything is drawn at 2× and scaled down for clean edges.
Change the parameters in the next cell and run all cells.

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageChops
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CMakeLists.txt").exists())
FONT = ROOT / "resources" / "fonts" / "Manrope-Regular.ttf"    # same weight as the "fjord." logo in the header

SIZE = 1024          # final size
SS = 2               # supersampling
N = SIZE * SS
HORIZON = 0.69       # water line (0 top … 1 bottom)

# Palette – Source/ui/Theme.h (polar night) and the aurora colours of the Play page visual
BG_TOP, BG_LOW = (13, 21, 32), (22, 38, 58)
AURORA = [(88, 209, 164), (63, 184, 200), (91, 143, 230)]     # green, teal, blue
MOUNTAIN_FAR, MOUNTAIN_NEAR = (30, 48, 72), (10, 17, 27)
WATER = (9, 15, 24)
INK = (226, 237, 240)                                          # icy white
DOT = (88, 209, 164)                                           # accent

yy, xx = np.mgrid[0:N, 0:N].astype(np.float32) / N

def show(img, size=320):
    display(img.resize((size, size), Image.LANCZOS))

def to_img(arr):
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def P(x, y):
    return (x * N, y * N)

## Sky

In [ ]:
t = np.clip(yy / HORIZON, 0, 1)[..., None]
sky = np.array(BG_TOP, np.float32) * (1 - t) + np.array(BG_LOW, np.float32) * t
show(to_img(sky))

## Aurora ribbons

Like the curtains in the plugin's fjord visual: a smooth sine-shaped top edge, a flat colour that fades out
downwards. No rays, no glow – just three overlapping translucent shapes.

In [ ]:
def ribbon(img, colour, base, amp, freq, phase, depth, alpha):
    top = base + amp * np.sin(2 * np.pi * freq * xx[0] + phase)          # y of the top edge per column
    d = (yy - top[None, :]) / depth                                       # 0 at the edge, 1 at the bottom
    edge = np.clip((yy - top[None, :]) * N / (1.2 * SS), 0, 1)           # anti-aliased top edge
    a = edge * np.clip(1 - d, 0, 1) ** 1.6 * alpha
    return img * (1 - a[..., None]) + np.array(colour, np.float32) * a[..., None]

art = sky.copy()
art = ribbon(art, AURORA[2], 0.19, 0.030, 0.8, 2.6, 0.17, 0.42)
art = ribbon(art, AURORA[1], 0.27, 0.040, 1.0, 0.9, 0.18, 0.50)
art = ribbon(art, AURORA[0], 0.35, 0.045, 1.2, 4.1, 0.20, 0.62)
show(to_img(art))

## Mountains and water

Two layers of straight-edged peaks fall towards the middle, where the fjord opens. Below the water line the
scene is mirrored and darkened, and a few rounded strokes stand for the reflections of the aurora.

In [ ]:
scene = to_img(art).convert("RGBA")
d = ImageDraw.Draw(scene)
h = HORIZON
far = [P(0, h), P(0, 0.50), P(0.10, 0.44), P(0.19, 0.52), P(0.30, 0.47), P(0.44, 0.62), P(0.50, h)]
far_r = [P(0.50, h), P(0.58, 0.58), P(0.70, 0.45), P(0.80, 0.51), P(0.90, 0.41), P(1, 0.48), P(1, h)]
near = [P(0, h), P(0, 0.40), P(0.08, 0.47), P(0.17, 0.38), P(0.36, h)]
near_r = [P(0.66, h), P(0.82, 0.43), P(0.90, 0.50), P(1, 0.36), P(1, h)]
for poly in (far, far_r):
    d.polygon(poly, fill=MOUNTAIN_FAR + (255,))
for poly in (near, near_r):
    d.polygon(poly, fill=MOUNTAIN_NEAR + (255,))

# water: the mirrored scene, mostly covered by the flat water colour
hz = int(h * N)
mirror = scene.crop((0, 2 * hz - N, N, hz)).transpose(Image.FLIP_TOP_BOTTOM)
flat = np.array(WATER + (255,), np.float32)
m = np.asarray(mirror, np.float32)
fade = (0.3 * np.clip(1 - np.arange(m.shape[0]) / (m.shape[0] * 0.75), 0, 1))[:, None, None]
scene.paste(to_img(flat * (1 - fade) + m * fade), (0, hz))

# reflection strokes: rounded dashes in the aurora colours, fainter towards the viewer
d = ImageDraw.Draw(scene)
# (y, half length, colour): centred under the mark, shorter and fainter towards the viewer
strokes = [(0.745, 0.30, 0), (0.795, 0.20, 1), (0.845, 0.12, 2), (0.895, 0.05, 0)]
w = 0.008 * N
for y, half, c in strokes:
    x0, x1 = 0.5 - half, 0.5 + half
    col = AURORA[c] + (int(255 * (0.75 - (y - h) * 2.6)),)
    d.rounded_rectangle((x0 * N, y * N - w / 2, x1 * N, y * N + w / 2), radius=w / 2, fill=col)
show(scene)

## The mark: "f."

Manrope Regular, like the logo in the plugin header: icy white, the dot in aurora green. The letter stands on the
water line and has a faint reflection.

In [ ]:
def mark(scene, scale=0.50, centre_x=0.49):
    out = scene.copy()
    font = ImageFont.truetype(str(FONT), int(N * scale))
    layer = Image.new("RGBA", out.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    l, t, r, b = d.textbbox((0, 0), "f", font=font)
    w = r - l
    dot_r = w * 0.15
    total_w = w + dot_r * 3.0
    x0 = centre_x * N - total_w / 2 - l
    y0 = HORIZON * N - b - 3 * SS            # stands on the water line
    d.text((x0, y0), "f", font=font, fill=INK + (255,))
    cx, cy = x0 + r + dot_r * 1.9, y0 + b - dot_r
    d.ellipse((cx - dot_r, cy - dot_r, cx + dot_r, cy + dot_r), fill=DOT + (255,))

    # soft shadow behind the mark so it stays clear in front of the ribbons
    shadow = Image.new("RGBA", out.size, (0, 0, 0, 0))
    shadow.putalpha(layer.getchannel("A").point(lambda a: int(a * 0.45)).filter(ImageFilter.GaussianBlur(10 * SS)))
    out = Image.alpha_composite(out, shadow)

    # reflection: mirrored at the water line, faint, fading out
    hz = int(HORIZON * N)
    refl = ImageChops.offset(layer.transpose(Image.FLIP_TOP_BOTTOM), 0, 2 * hz - N)
    a = np.asarray(refl, np.float32).copy()
    rows = np.arange(N, dtype=np.float32)[:, None]
    a[..., 3] *= np.clip(1 - (rows - hz) / (N * 0.2), 0, 1) * (rows >= hz) * 0.22
    out = Image.alpha_composite(out, Image.fromarray(a.astype(np.uint8)))
    return Image.alpha_composite(out, layer)

final = mark(scene)
show(final)

## Export

- **Icon**: macOS grid – the body is 824² inside 1024² with continuous (superellipse) corners, a hairline rim
  and a soft drop shadow in the transparent margin.
- **Profile picture**: full bleed; platforms crop it to a circle, so the mark sits in the centre.

In [ ]:
def squircle_mask(size, body, n=5.0):
    """superellipse |x|^n + |y|^n = 1 – the continuous corners of macOS icons"""
    g = (np.mgrid[0:size, 0:size].astype(np.float32) + 0.5 - size / 2) / (body / 2)
    v = np.abs(g[0]) ** n + np.abs(g[1]) ** n
    return np.clip((1.0 - v) * body / 6, 0, 1)

def make_icon(art):
    body = int(824 * SS)
    tile = Image.new("RGBA", (N, N), (0, 0, 0, 0))
    off = (N - body) // 2
    tile.paste(art.resize((body, body), Image.LANCZOS), (off, off))
    mask = squircle_mask(N, body)
    rim = np.clip(mask - squircle_mask(N, body - 3 * SS), 0, 1)
    rgba = np.asarray(tile, np.float32).copy()
    rgba[..., :3] += rim[..., None] * np.array([45, 70, 85], np.float32)
    rgba[..., 3] = mask * 255
    icon = to_img(rgba)
    shadow = Image.new("RGBA", icon.size, (0, 0, 0, 0))
    shadow.putalpha(Image.fromarray((mask * 255 * 0.4).astype(np.uint8)).filter(ImageFilter.GaussianBlur(14 * SS)))
    out = Image.alpha_composite(ImageChops.offset(shadow, 0, 10 * SS), icon)
    return out.resize((SIZE, SIZE), Image.LANCZOS)

icon = make_icon(final)
profile = final.convert("RGB").resize((SIZE, SIZE), Image.LANCZOS)
icon.save(ROOT / "resources" / "icon.png")
profile.save(ROOT / "resources" / "profile.png")
print("written: resources/icon.png, resources/profile.png")

# preview: icon on light and dark, profile as a circle, small sizes
def on(bg, img, s=300):
    return Image.alpha_composite(Image.new("RGBA", (s, s), bg), img.resize((s, s), Image.LANCZOS))
circle = Image.new("L", (300, 300), 0); ImageDraw.Draw(circle).ellipse((0, 0, 299, 299), fill=255)
round_profile = Image.new("RGBA", (300, 300), (237, 243, 244, 255))
round_profile.paste(profile.resize((300, 300), Image.LANCZOS), (0, 0), circle)
sheet = Image.new("RGBA", (300 * 3 + 152, 300), (237, 243, 244, 255))
sheet.paste(on((237, 243, 244, 255), icon), (0, 0))
sheet.paste(on((13, 21, 32, 255), icon), (300, 0))
sheet.paste(round_profile, (600, 0))
x = 920
for s in (64, 32, 16):
    sheet.paste(on((237, 243, 244, 255), icon, s), (x, 150 - s // 2)); x += s + 8
display(sheet)